# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**1. Unit of Analysis:**
One row = **One unique content page (uniquely identified by `content_hash_id`) for a specific client (`client_hash_id`) in a given month.**

**2. Tables used:**
- `dim_clients` (to check history start dates and GSC/GA4 availability)
- `dim_content` (to fetch content metadata like content age, word count, cpc)
- `fact_content_daily_performance` (to aggregate time-series features and target labels)
- `fact_content_query_90d` (to join query-level context signals)

**3. Time Window:**
- **Feature window:** The first 15 days of the month (e.g., `report_date <= '2026-03-15'`).
- **Label window (target):** The remaining days of the month (e.g., `report_date >= '2026-03-16'`).
- Split is done cleanly at day 15 to ensure all features are measured strictly before the label period starts, preventing future information leakage.

**4. What we predict / rank (Target/Proxy):**
We predict `is_declining`, a binary proxy label indicating whether a page's impressions in the label window drop by more than 20% compared to the feature window (i.e. `imp_label_window < 0.8 * imp_feature_window`).

**5. Deliberately Excluded:**
- **Product Flags & Pre-baked labels:** We exclude product flags like `health_score` or pre-made status flags from `dim_content` because they represent decisions already made by other parts of the system, creating circular logic.
- **Future metrics:** Any click/impression metrics from the label window (`report_date >= '2026-03-16'`) are excluded from the feature set to prevent information leakage.


In [1]:
# Verify that (client_hash_id, content_hash_id, report_date) is unique at the daily level
import duckdb
import os, getpass

# Fetch the token from the environment variable (best practice) or prompt the user safely
token = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET hf_http (TYPE http, BEARER_TOKEN '{token}')")

REL = 'https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main'
FACT_DAILY_2026_03 = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/data_0.parquet')"

print("=== Fact 1: Grain verification (client + content + date uniqueness) ===")
grain_query = f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) as c
    FROM {FACT_DAILY_2026_03}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
"""
grain_df = con.sql(grain_query).df()
print("Duplicate rows count:", len(grain_df))
if len(grain_df) == 0:
    print("Verified: One row per page-day grain holds perfectly!")


=== Fact 1: Grain verification (client + content + date uniqueness) ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate rows count: 0
Verified: One row per page-day grain holds perfectly!


## 2. Fields: feature / label / context / excluded

We classify the fields we plan to touch in `month=2026-03` into the four required buckets:

- **Feature Bucket (Knowable before the prediction moment):**
  - `imp_feat`: Sum of impressions in the first 15 days of March.
  - `clicks_feat`: Sum of clicks in the first 15 days of March.
  - `avg_pos_feat`: Average search position in the first 15 days of March.
  - `scroll_rate_feat`: Sum of scroll events divided by pageviews in the first 15 days of March.
  - `ai_sessions_feat`: Sum of organic sessions referred from AI in the first 15 days of March.
- **Label / Proxy Bucket (Calculated from future outcomes):**
  - `is_declining`: Computed as `(imp_label_window < 0.8 * imp_feat): 1 else 0`.
- **Context Bucket (Used only for splitting, grouping, and joining):**
  - `client_hash_id`: Pseudonymized client identifier (crucial for client-holdout validation).
  - `content_hash_id`: Pseudonymized content page identifier.
- **Excluded Bucket (Excluded to prevent circular logic and leakage):**
  - `imp_label_window`: Excluded because it is measured *during* the label period and directly computes the label.
  - `clicks_label_window`: Excluded because it is a future outcome during the label period.
  - `provider_used` & `model_used`: Excluded as content properties to avoid learning vendor-specific signatures.


In [2]:
# Show the columns present in our primary fact table for reference
df_cols = con.sql(f"DESCRIBE SELECT * FROM {FACT_DAILY_2026_03}").df()
print(df_cols[['column_name', 'column_type']].to_string(index=False))


             column_name column_type
             report_date        DATE
          client_hash_id     VARCHAR
         content_hash_id     VARCHAR
          client_has_gsc     BOOLEAN
          client_has_ga4     BOOLEAN
      gsc_data_available     BOOLEAN
      ga4_data_available     BOOLEAN
         gsc_impressions      BIGINT
              gsc_clicks      BIGINT
        gsc_sum_position      BIGINT
        gsc_avg_position      DOUBLE
           ga4_pageviews      BIGINT
            ga4_sessions      BIGINT
               ga4_users      BIGINT
    ga4_engaged_sessions      BIGINT
ga4_total_engagement_sec      BIGINT
        sessions_organic      BIGINT
         sessions_direct      BIGINT
       sessions_referral      BIGINT
         sessions_social      BIGINT
           sessions_paid      BIGINT
             sessions_ai      BIGINT
              ai_chatgpt      BIGINT
           ai_perplexity      BIGINT
               ai_gemini      BIGINT
              ai_copilot      BIGINT
 

## 3. Verify it with queries (grain, counts, missing values, windows)

We run three queries to verify the contract claims on the mid-panel month `month=2026-03`:
1. **Grain Check:** Confirm that client + content + report_date is unique.
2. **Counts and Date Span Check:** Verify the row count and min/max dates in March 2026.
3. **Availability Check:** Show how many rows survive when GSC and GA4 available flags are `TRUE`.
4. **Features + Trap Experiment:** Build the feature frame, introduce a leaked column (`imp_label_window`) to show perfect accuracy, then train the honest model without it.


In [3]:
# Query 1 & 2: Row count, Date range, and Grain check
print("=== Grain and Counts Check ===")
stats = con.sql(f"""
    SELECT 
        COUNT(*) as total_rows, 
        MIN(report_date) as min_date, 
        MAX(report_date) as max_date,
        COUNT(DISTINCT (client_hash_id || '-' || content_hash_id || '-' || report_date)) as unique_grain_count
    FROM {FACT_DAILY_2026_03}
""").df()
print(stats.to_string(index=False))

# Query 3: Availability check using IS TRUE
print("\n=== Availability Check ===")
availability = con.sql(f"""
    SELECT 
        COUNT(*) as total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) as gsc_avail_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as ga4_avail_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE THEN 1 ELSE 0 END) as both_avail_rows
    FROM {FACT_DAILY_2026_03}
""").df()
print(availability.to_string(index=False))

# Build Features & Leakage Trap Experiment
print("\n=== Five Features + Trap Experiment ===")
query = f"""
    WITH stats AS (
        SELECT 
            client_hash_id,
            content_hash_id,
            -- Features (knowable before March 16)
            SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_feat,
            SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_clicks ELSE 0 END) AS clicks_feat,
            AVG(CASE WHEN report_date <= '2026-03-15' THEN gsc_avg_position ELSE NULL END) AS avg_pos_feat,
            SUM(CASE WHEN report_date <= '2026-03-15' THEN scroll_events ELSE 0 END) AS scroll_feat,
            SUM(CASE WHEN report_date <= '2026-03-15' THEN ga4_pageviews ELSE 0 END) AS pageviews_feat,
            SUM(CASE WHEN report_date <= '2026-03-15' THEN sessions_ai ELSE 0 END) AS ai_sessions_feat,
            
            -- Future Outcomes (used to define the label, and as the LEAK feature)
            SUM(CASE WHEN report_date >= '2026-03-16' THEN gsc_impressions ELSE 0 END) AS imp_label_window
        FROM {FACT_DAILY_2026_03}
        GROUP BY 1, 2
        HAVING imp_feat >= 100
    )
    SELECT 
        client_hash_id,
        content_hash_id,
        imp_feat,
        clicks_feat,
        COALESCE(avg_pos_feat, 0) AS avg_pos_feat,
        CASE WHEN pageviews_feat > 0 THEN (scroll_feat * 100.0 / pageviews_feat) ELSE 0 END AS scroll_rate_feat,
        ai_sessions_feat,
        imp_label_window,
        CASE WHEN imp_label_window < 0.8 * imp_feat THEN 1 ELSE 0 END AS is_declining
    FROM stats
"""
df_model = con.sql(query).df()
print(f"Aggregated data shape: {df_model.shape}")
print(f"Base rate of decline: {df_model['is_declining'].mean():.1%}")

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

train_df, test_df = train_test_split(df_model, test_size=0.3, random_state=42, stratify=df_model['is_declining'])

# Trap (Leakage) Run
print("\n--- Leakage Run ---")
leak_features = ['imp_feat', 'clicks_feat', 'avg_pos_feat', 'scroll_rate_feat', 'ai_sessions_feat', 'imp_label_window']
model_leak = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(train_df[leak_features], train_df['is_declining'])
print(classification_report(test_df['is_declining'], model_leak.predict(test_df[leak_features]), digits=3))

# Honest Run
print("\n--- Honest Run ---")
honest_features = ['imp_feat', 'clicks_feat', 'avg_pos_feat', 'scroll_rate_feat', 'ai_sessions_feat']
model_honest = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(train_df[honest_features], train_df['is_declining'])
print(classification_report(test_df['is_declining'], model_honest.predict(test_df[honest_features]), digits=3))


=== Grain and Counts Check ===


 total_rows   min_date   max_date  unique_grain_count
    9841378 2026-03-01 2026-03-31             9841378

=== Availability Check ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  gsc_avail_rows  ga4_avail_rows  both_avail_rows
    9841378       3611061.0        413966.0         364347.0

=== Five Features + Trap Experiment ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Aggregated data shape: (77540, 9)
Base rate of decline: 28.5%



--- Leakage Run ---


              precision    recall  f1-score   support

           0      0.985     0.995     0.990     16621
           1      0.987     0.961     0.974      6641

    accuracy                          0.985     23262
   macro avg      0.986     0.978     0.982     23262
weighted avg      0.985     0.985     0.985     23262


--- Honest Run ---


              precision    recall  f1-score   support

           0      0.733     0.857     0.790     16621
           1      0.379     0.218     0.277      6641

    accuracy                          0.675     23262
   macro avg      0.556     0.538     0.533     23262
weighted avg      0.632     0.675     0.644     23262



## 4. Data limits

**Named Limitations of our slice (`month=2026-03`):**
1. **Unbalanced History Depth:** Clients have different history start dates. For example, some clients have GSC history going back to 2025-01, while others only start in 2026. Applying a strict 90-day time-series window dynamically reduces the number of eligible clients and pages because younger clients do not satisfy the minimum history depth.
2. **Analytics Availability Gap:** Our availability check shows that while GSC (search performance) data is available for **36.7%** (3,611,061 rows) of the daily fact table, GA4 (analytics engagement) is only available for **4.2%** (413,966 rows). If we force our model to rely heavily on GA4 metrics (like `scroll_rate` or `ga4_pageviews`), we instantly drop 90% of our available data. A robust modeling design must handle this systematic missingness cleanly (e.g. by using `gsc_data_available` and `ga4_data_available` has-flags rather than a blind `fillna(0)`).
3. **Window Overlap and Leakage Trap:** Since `fact_content_query_90d` contains metrics aggregated over a rolling 90-day window, utilizing its traffic columns directly when predicting a monthly label leads to leakage. We must ensure that query-level signals are aligned with a historical window before prediction time.


In [4]:
# Confirm GSC vs GA4 availability gaps across dim_clients
client_history = con.sql(f"""
    SELECT 
        COUNT(client_hash_id) as total_clients,
        SUM(CASE WHEN has_gsc_access IS TRUE THEN 1 ELSE 0 END) as clients_with_gsc,
        SUM(CASE WHEN has_ga4_access IS TRUE THEN 1 ELSE 0 END) as clients_with_ga4,
        SUM(CASE WHEN has_gsc_access IS TRUE AND has_ga4_access IS TRUE THEN 1 ELSE 0 END) as clients_with_both
    FROM read_parquet('{REL}/dim_clients.parquet')
""").df()
print("Client-level GSC and GA4 accessibility:")
print(client_history.to_string(index=False))


Client-level GSC and GA4 accessibility:
 total_clients  clients_with_gsc  clients_with_ga4  clients_with_both
           104              67.0              54.0               53.0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
